In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, precision_recall_fscore_support

In [2]:

df = pd.read_excel("/content/Data_new_version.xlsx")

In [3]:
df

,Campaign_ID,Brand,Campaign_Type,Target_Audience,Channel_Used,Duration,Language,Customer_Segment,Campaign_Date,Impressions,...,Conversion_Rate,ROI,Engagement_Score,ROI_Category,Campaign_Month,Campaign_Quater,Primary_Channel,Channel_Count,Duration_Group,Revenue_Per_Conversion
0,NY-CMP-13007,Nykaa,Paid Ads,Youth,"Email, YouTube, Facebook",29,Hindi,Tier 2 City Customers,2025-02-14,99483,...,0.7678,74.42,30.17,Excellent,2025-02,Q1-2025,Email,3,Very Long (22+ days),685
1,NY-CMP-22689,Nykaa,Influencer,Youth,"Instagram, YouTube",20,Hindi,College Students,2025-01-11,97281,...,0.7095,66.32,27.26,Excellent,2025-01,Q1-2025,Instagram,2,Long (15-21 days),719
2,NY-CMP-49715,Nykaa,Paid Ads,Premium Shoppers,WhatsApp,12,Tamil,Working Women,2025-05-22,89428,...,0.6881,65.00,25.54,Excellent,2025-05,Q2-2025,WhatsApp,1,Medium (8-14 days),797
3,NY-CMP-26356,Nykaa,Paid Ads,Premium Shoppers,Facebook,5,Hindi,Youth,2025-06-15,90880,...,0.7316,64.14,27.06,Excellent,2025-06,Q2-2025,Facebook,1,Short (1-7 days),706
4,NY-CMP-39329,Nykaa,Paid Ads,Youth,"Google, Facebook, YouTube",27,Hindi,Working Women,2024-11-06,77578,...,0.7376,59.64,27.64,Excellent,2024-11,Q4-2024,Google,3,Very Long (22+ days),728
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
165520,TI-CMP-16012,Tira,Paid Ads,Youth,Google,6,Bengali,Premium Shoppers,2025-05-17,26396,...,0.3043,-0.97,2.58,Poor,2025-05,Q2-2025,Google,1,Short (1-7 days),275
165521,TI-CMP-56201,Tira,Email,Youth,"Facebook, WhatsApp, YouTube",14,Tamil,Premium Shoppers,2024-08-17,12741,...,0.2881,-0.97,2.65,Poor,2024-08,Q3-2024,Facebook,3,Medium (8-14 days),411
165522,TI-CMP-8939,Tira,Social Media,Youth,"Email, WhatsApp, Instagram",11,English,Working Women,2025-03-04,13551,...,0.4405,-0.97,3.29,Poor,2025-03,Q1-2025,Email,3,Medium (8-14 days),235
165523,TI-CMP-20738,Tira,Influencer,Premium Shoppers,"Instagram, WhatsApp, Email",6,Tamil,Tier 2 City Customers,2025-02-08,11146,...,0.3099,-0.98,3.67,Poor,2025-02,Q1-2025,Instagram,3,Short (1-7 days),276


In [4]:
FEATURES_V1 = ["Brand", "Campaign_Type", "Target_Audience",
              "Primary_Channel", "Duration_Group", "Language",
              "Customer_Segment", "Channel_Count", "Duration"]
FEATURES_V2 = FEATURES_V1 + ["Impressions", "Engagement_Score"]
FEATURES_V3 = FEATURES_V2 + ["CPA", "CTR", "Conversion_Rate"]

In [5]:
def prep_xy(df, feats):
  X = pd.get_dummies(df[feats], drop_first=True)
  y = df["Is_Poor"].values
  return X, y

In [6]:
df["Is_Poor"] = (df["ROI"] < 0).astype(int)

In [9]:
print("Số lượng chiến dịch Poor (1) và OK (0):")
print(df["Is_Poor"].value_counts())

Số lượng chiến dịch Poor (1) và OK (0):
Is_Poor
0    126267
1     39258
Name: count, dtype: int64


In [19]:
X1, y = prep_xy(df, FEATURES_V1)
X1_tr, X1_te, y_tr, y_te = train_test_split(X1, y, test_size=0.2, random_state=42)

def train_logit(Xtr, Xte, ytr, yte, label):
  m = LogisticRegression(class_weight="balanced",
                        max_iter=1000, random_state=42)
  m.fit(Xtr, ytr)
  p = m.predict_proba(Xte)[:, 1] # xác suất P(Poor)
  yp = (p >= 0.5).astype(int) # ngưỡng mặc định 0.5
  auc = roc_auc_score(yte, p)
  pr, rc, f1, _ = precision_recall_fscore_support(yte, yp,
  average="binary", pos_label=1)
  print(f"{label}: AUC={auc:.3f} P={pr:.3f} R={rc:.3f} F1={f1:.3f}")
  return m, p, yp

In [20]:
train_logit(X1_tr, X1_te, y_tr, y_te, "Version 1");

Version 1: AUC=0.496 P=0.237 R=0.498 F1=0.321


In [21]:
X2, y = prep_xy(df, FEATURES_V2)
X2_tr, X2_te, y_tr, y_te = train_test_split(X2, y, test_size=0.2, random_state=42)

In [22]:
train_logit(X2_tr, X2_te, y_tr, y_te, "Version 2");

Version 2: AUC=0.885 P=0.544 R=0.798 F1=0.647


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [23]:
X3, y = prep_xy(df, FEATURES_V3)
X3_tr, X3_te, y_tr, y_te = train_test_split(X3, y, test_size=0.2, random_state=42)

In [24]:
train_logit(X3_tr, X3_te, y_tr, y_te, "Version 3");

Version 3: AUC=0.966 P=0.743 R=0.870 F1=0.801


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [25]:
# Quét ngưỡng CPA từ 200 đến 1000 để tìm F1 tốt nhất
def cpa_threshold_sweep(df, thresholds=(200, 300, 400, 500, 600, 700, 800, 900, 1000)):
    rows = []
    truly = df["Is_Poor"].values
    for T in thresholds:
        flagged = (df["CPA"] > T).values
        tp = ((flagged == 1) & (truly == 1)).sum()
        fp = ((flagged == 1) & (truly == 0)).sum()
        fn = ((flagged == 0) & (truly == 1)).sum()
        pr = tp / (tp + fp) if tp + fp > 0 else 0
        rc = tp / (tp + fn) if tp + fn > 0 else 0
        f1 = 2 * pr * rc / (pr + rc) if pr + rc > 0 else 0
        rows.append((T, int(flagged.sum()), round(pr, 3), round(rc, 3), round(f1, 3)))
    return pd.DataFrame(rows, columns=["Ngưỡng CPA", "Số Campaign bị gắn cờ", "Precision", "Recall", "F1"])

# Chạy hàm
cpa_threshold_sweep(df)

,Ngưỡng CPA,Số Campaign bị gắn cờ,Precision,Recall,F1
0,200,85377,0.460,1.000,0.630
1,300,60255,0.620,0.951,0.751
2,400,44752,0.753,0.858,0.802
3,500,34391,0.859,0.752,0.802
4,600,27078,0.933,0.644,0.762
5,700,21634,0.983,0.541,0.698
6,800,17665,1.000,0.450,0.621
7,900,14715,1.000,0.375,0.545
8,1000,12313,1.000,0.314,0.478
